# JATAYU: Fine-Tuning YOLOv8x on xBD Disaster Damage Dataset
### Hackathon: VISTERA 2026 | Team: IGNITE
This notebook trains YOLOv8x on building damage pairs from the xBD benchmark on free Google Colab / Kaggle T4 GPU.

**License & Terms:**
xBD Dataset is licensed under Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International (CC BY-NC-SA 4.0).
Registration required at https://xview2.org.

In [ ]:
# Step 1: Install dependencies
!pip install ultralytics shapely albumentations

In [ ]:
# Step 2: Convert xBD polygons to YOLO format
!python scripts/prepare_xbd_to_yolo.py --input-dir datasets/xbd/labels --output-dir datasets/yolo_xbd/labels/train

In [ ]:
# Step 3: Train YOLOv8x with sensor and region shift augmentations
from ultralytics import YOLO

model = YOLO('yolov8x.pt')
results = model.train(
    data='datasets/xbd.yaml',
    epochs=50,
    imgsz=1024,
    batch=16,
    mosaic=0.8,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=90.0,
    scale=0.5,
    project='runs/train',
    name='jatayu_yolov8x_xbd'
)

In [ ]:
# Step 4: Evaluate test split and export metrics.json
val_metrics = model.val(data='datasets/xbd.yaml', split='test')
print(f'Overall Precision: {val_metrics.box.mp:.3f}')
print(f'Overall Recall: {val_metrics.box.mr:.3f}')
print(f'Overall mAP50: {val_metrics.box.map50:.3f}')
print(f'Confusion Matrix:\n{val_metrics.confusion_matrix.matrix}')